# LSTM Autoencoder for Anomaly Detection

This notebook implements an **LSTM Autoencoder** for anomaly detection on SWaT per-PLC stage data. It serves as the deep-learning baseline for comparing different anomaly detection methods.

## Architecture
The model consists of two components:
1. **Encoder**: An LSTM network that processes the input sequence window of length `T` and compresses it into a fixed-size latent representation (bottleneck). It uses the final hidden state of the LSTM.
2. **Decoder**: Another LSTM network that reconstructs the original sequence from the latent vector. A common seq2seq pattern is used here, where the latent vector is repeated `T` times as input to the decoder.

## Why Reconstruction Error?
Autoencoders are trained to minimize the reconstruction error on normal data. During inference, they can effectively reconstruct patterns they have seen during training. However, when presented with anomalous data (e.g., attacks), the model fails to reconstruct it accurately, resulting in a higher reconstruction error (MSE). By thresholding this error, we can identify anomalies.

## Edge-Deployment Tradeoffs
Unlike lighter-weight models in this project (such as Markov chains, PASAD, Physical Invariants, or EWMA/CUSUM), neural networks like this LSTM Autoencoder have a **significantly higher computational and memory cost**. Edge deployment on PLCs or local network gateways might be constrained by CPU, memory, and energy. This notebook explicitly measures inference latency (mean and p95) and memory footprint to evaluate these tradeoffs.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import joblib

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# 1. Configuration / Parameters
STAGE_NUM = 1                # PLC Stage to analyze (1 to 6)
T = 60                       # Sequence window length (timesteps)
STRIDE_TRAIN = 5             # Stride for training windows to speed up training
STRIDE_EVAL = 1              # Stride for evaluation windows (must be 1 for exact timestep alignment or handled in aggregation)
LSTM_HIDDEN_SIZE = 32        # Number of hidden units in LSTM
LATENT_SIZE = 8              # Bottleneck latent vector size
NUM_LAYERS = 1               # Number of LSTM layers (encoder/decoder)
BATCH_SIZE = 128             # Training batch size
EPOCHS = 50                  # Maximum number of epochs
LEARNING_RATE = 1e-3         # Optimizer learning rate
PATIENCE = 5                 # Early stopping patience
THRESHOLD_MULTIPLIER = 3.0   # Anomaly threshold: mean(val_mse) + THRESHOLD_MULTIPLIER * std(val_mse)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create models directory
os.makedirs('models', exist_ok=True)


In [ ]:
# 2. Preprocessing Functions
def load_and_clean_data(stage_num):
    # Load normal and attack data for the given stage
    normal_df = pd.read_csv(f'dataset/normal_stage{stage_num}.csv')
    attack_df = pd.read_csv(f'dataset/attack_stage{stage_num}.csv')
    
    # Strip column names
    normal_df.columns = normal_df.columns.str.strip()
    attack_df.columns = attack_df.columns.str.strip()
    
    # Parse timestamp and sort
    if 'Timestamp' in normal_df.columns:
        normal_df['Timestamp'] = pd.to_datetime(normal_df['Timestamp'].str.strip(), format='mixed', dayfirst=True)
        normal_df = normal_df.sort_values('Timestamp').reset_index(drop=True)
    if 'Timestamp' in attack_df.columns:
        attack_df['Timestamp'] = pd.to_datetime(attack_df['Timestamp'].str.strip(), format='mixed', dayfirst=True)
        attack_df = attack_df.sort_values('Timestamp').reset_index(drop=True)
        
    # Numeric coercion and forward-fill NaNs
    cols_to_coerce_normal = [c for c in normal_df.columns if c not in ['Timestamp', 'attack']]
    for c in cols_to_coerce_normal:
        normal_df[c] = pd.to_numeric(normal_df[c], errors='coerce')
    normal_df[cols_to_coerce_normal] = normal_df[cols_to_coerce_normal].ffill().bfill()

    cols_to_coerce_attack = [c for c in attack_df.columns if c not in ['Timestamp', 'attack']]
    for c in cols_to_coerce_attack:
        attack_df[c] = pd.to_numeric(attack_df[c], errors='coerce')
    attack_df[cols_to_coerce_attack] = attack_df[cols_to_coerce_attack].ffill().bfill()
    
    return normal_df, attack_df

def scale_data(normal_df, attack_df):
    features = [c for c in normal_df.columns if c not in ['Timestamp', 'attack']]
    scaler = MinMaxScaler()
    
    # Fit ONLY on normal data
    scaler.fit(normal_df[features])
    
    # Transform both
    normal_scaled = scaler.transform(normal_df[features])
    attack_scaled = scaler.transform(attack_df[features])
    
    return normal_scaled, attack_scaled, features, scaler

def create_windows(data, T, stride):
    windows = []
    for i in range(0, len(data) - T + 1, stride):
        windows.append(data[i:i+T])
    return np.array(windows)


In [ ]:
# 3. Model Architecture
class EncoderLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, latent_size, num_layers=1):
        super(EncoderLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, latent_size)
        
    def forward(self, x):
        # x shape: (batch, seq_len, features)
        _, (h_n, _) = self.lstm(x)
        # h_n shape: (num_layers, batch, hidden_size)
        h_n_last = h_n[-1] # take the last layer's hidden state
        latent = self.fc(h_n_last)
        return latent

class DecoderLSTM(nn.Module):
    def __init__(self, latent_size, hidden_size, output_size, num_layers=1):
        super(DecoderLSTM, self).__init__()
        self.lstm = nn.LSTM(latent_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x, seq_len):
        # x is the latent vector: (batch, latent_size)
        # Repeat it seq_len times
        x = x.unsqueeze(1).repeat(1, seq_len, 1)
        # x shape: (batch, seq_len, latent_size)
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, latent_size, num_layers=1):
        super(LSTMAutoencoder, self).__init__()
        self.encoder = EncoderLSTM(input_size, hidden_size, latent_size, num_layers)
        self.decoder = DecoderLSTM(latent_size, hidden_size, input_size, num_layers)
        
    def forward(self, x):
        seq_len = x.size(1)
        latent = self.encoder(x)
        reconstructed = self.decoder(latent, seq_len)
        return reconstructed


In [ ]:
# 4. Training Loop
def train_model(model, train_loader, val_loader, epochs, lr, patience):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch_x in train_loader:
            batch_x = batch_x[0].to(device)
            optimizer.zero_grad()
            reconstructed = model(batch_x)
            loss = criterion(reconstructed, batch_x)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_x.size(0)
            
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x in val_loader:
                batch_x = batch_x[0].to(device)
                reconstructed = model(batch_x)
                loss = criterion(reconstructed, batch_x)
                val_loss += loss.item() * batch_x.size(0)
                
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)
        
        #print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                #print("Early stopping triggered.")
                break
                
    model.load_state_dict(best_model_state)
    return train_losses, val_losses, model

def plot_losses(train_losses, val_losses, stage_num):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title(f'Training and Validation Loss - Stage {stage_num}')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# 5. Anomaly Scoring & Evaluation
def calculate_reconstruction_errors(model, dataloader):
    model.eval()
    errors = []
    criterion = nn.MSELoss(reduction='none') # per element loss
    
    with torch.no_grad():
        for batch_x in dataloader:
            batch_x = batch_x[0].to(device)
            reconstructed = model(batch_x)
            # MSE per window per timestep
            loss = criterion(reconstructed, batch_x).mean(dim=-1) # Mean over features, shape (batch, T)
            errors.append(loss.cpu().numpy())
            
    return np.concatenate(errors, axis=0) # shape (num_windows, T)

def aggregate_window_errors_to_timesteps(window_errors, data_len, T, stride):
    # window_errors shape: (num_windows, T)
    timestep_errors = np.zeros(data_len)
    timestep_counts = np.zeros(data_len)
    
    num_windows = window_errors.shape[0]
    for i in range(num_windows):
        start_idx = i * stride
        end_idx = start_idx + T
        timestep_errors[start_idx:end_idx] += window_errors[i]
        timestep_counts[start_idx:end_idx] += 1
        
    # Average across overlapping windows
    timestep_counts[timestep_counts == 0] = 1 # Avoid division by zero
    return timestep_errors / timestep_counts

def evaluate_predictions(y_true, y_pred, y_scores):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    metrics = {
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_scores),
        'FPR': fpr,
        'FNR': fnr
    }
    return metrics, cm


In [ ]:
# 6. Latency & Size Benchmarking
def benchmark_model(model, T, input_size):
    model.eval()
    model.to('cpu') # Benchmark on CPU as requested for edge deployment constraints
    
    dummy_input = torch.randn(1, T, input_size).to('cpu')
    
    # Warmup
    for _ in range(10):
        _ = model(dummy_input)
        
    latencies = []
    with torch.no_grad():
        for _ in range(100):
            start = time.perf_counter()
            _ = model(dummy_input)
            end = time.perf_counter()
            latencies.append((end - start) * 1000) # ms
            
    mean_latency = np.mean(latencies)
    p95_latency = np.percentile(latencies, 95)
    
    # Parameters and memory
    param_count = sum(p.numel() for p in model.parameters())
    
    # Approximate state_dict size in MB (float32 = 4 bytes per param)
    mem_footprint_mb = (param_count * 4) / (1024 * 1024)
    
    model.to(device) # move back
    return mean_latency, p95_latency, param_count, mem_footprint_mb


In [ ]:
# 7. Run Pipeline for a Single Stage
def run_stage(stage_num, plot=False):
    print(f"\n--- Processing Stage {stage_num} ---")
    
    # 1. Load Data
    normal_df, attack_df = load_and_clean_data(stage_num)
    
    # 2. Scale
    normal_scaled, attack_scaled, features, scaler = scale_data(normal_df, attack_df)
    input_size = len(features)
    
    # 3. Create Windows
    normal_windows = create_windows(normal_scaled, T, STRIDE_TRAIN)
    attack_windows_eval = create_windows(attack_scaled, T, STRIDE_EVAL)
    
    # Split normal into Train/Val (90/10)
    split_idx = int(len(normal_windows) * 0.9)
    train_windows = normal_windows[:split_idx]
    val_windows = normal_windows[split_idx:]
    
    train_tensor = torch.tensor(train_windows, dtype=torch.float32)
    val_tensor = torch.tensor(val_windows, dtype=torch.float32)
    attack_tensor = torch.tensor(attack_windows_eval, dtype=torch.float32)
    
    train_loader = DataLoader(TensorDataset(train_tensor), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(val_tensor), batch_size=BATCH_SIZE, shuffle=False)
    attack_loader = DataLoader(TensorDataset(attack_tensor), batch_size=BATCH_SIZE, shuffle=False)
    
    # 4. Model Setup
    model = LSTMAutoencoder(input_size, LSTM_HIDDEN_SIZE, LATENT_SIZE, NUM_LAYERS).to(device)
    
    # 5. Training
    train_losses, val_losses, model = train_model(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, PATIENCE)
    
    if plot:
        plot_losses(train_losses, val_losses, stage_num)
        
    # Save Model and Scaler
    torch.save(model.state_dict(), f'models/lstm_ae_stage{stage_num}.pt')
    joblib.dump(scaler, f'models/scaler_stage{stage_num}.pkl')
    
    # 6. Anomaly Scoring on Validation Set to find threshold
    val_loader_eval = DataLoader(TensorDataset(val_tensor), batch_size=BATCH_SIZE, shuffle=False)
    val_window_errors = calculate_reconstruction_errors(model, val_loader_eval)
    val_timestep_errors = aggregate_window_errors_to_timesteps(val_window_errors, len(val_scaled_data) if 'val_scaled_data' in locals() else len(normal_scaled[split_idx*STRIDE_TRAIN:]), T, STRIDE_TRAIN) # Approximation for threshold
    
    # Actually, a better threshold is calculated directly from val_window_errors flattened
    threshold = np.mean(val_window_errors) + THRESHOLD_MULTIPLIER * np.std(val_window_errors)
    
    # 7. Evaluation on Attack Set
    attack_window_errors = calculate_reconstruction_errors(model, attack_loader)
    attack_timestep_errors = aggregate_window_errors_to_timesteps(attack_window_errors, len(attack_df), T, STRIDE_EVAL)
    
    metrics = {}
    if stage_num == 6:
        # Stage 6: skip supervised metrics, report stats
        metrics = {
            'Mean Error': np.mean(attack_timestep_errors),
            'Max Error': np.max(attack_timestep_errors),
            'Min Error': np.min(attack_timestep_errors)
        }
        if plot:
            plt.figure(figsize=(12, 4))
            plt.plot(attack_timestep_errors, label='Reconstruction Error')
            plt.axhline(y=threshold, color='r', linestyle='--', label='Threshold')
            plt.title('Stage 6 Reconstruction Errors (No Ground Truth Labels used)')
            plt.legend()
            plt.show()
    else:
        y_true = attack_df['attack'].fillna(0).values
        # Binarize labels
        y_true = (y_true > 0).astype(int)
        y_scores = attack_timestep_errors
        y_pred = (y_scores > threshold).astype(int)
        
        metrics, cm = evaluate_predictions(y_true, y_pred, y_scores)
        print("Confusion Matrix:")
        print(cm)
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, zero_division=0))
        
    # 8. Benchmarking
    mean_lat, p95_lat, params, mem_mb = benchmark_model(model, T, input_size)
    metrics['Mean Latency (ms)'] = mean_lat
    metrics['P95 Latency (ms)'] = p95_lat
    metrics['Parameters'] = params
    metrics['Mem Footprint (MB)'] = mem_mb
    
    return metrics


In [ ]:
# Run for configured STAGE_NUM
if __name__ == '__main__':
    metrics = run_stage(STAGE_NUM, plot=True)
    print("Metrics for Stage", STAGE_NUM)
    for k, v in metrics.items():
        print(f"{k}: {v}")


In [ ]:
# 8. Loop Across All Stages
all_metrics = []

for stage in range(1, 7):
    m = run_stage(stage, plot=False)
    m['Stage'] = stage
    all_metrics.append(m)
    
results_df = pd.DataFrame(all_metrics)
# Reorder columns
cols = ['Stage'] + [c for c in results_df.columns if c != 'Stage']
results_df = results_df[cols]

print("\n=== Summary Across All Stages ===")
display(results_df)
